# Manufacturing Defect Rate Analysis

## Problem Statement

A manufacturing analytics team wants to understand how products rank by revenue within their respective categories.

For each product that has a corresponding sales record, return the product category, product name, revenue, and rank within the category.

## Input Tables

### manufacture_product

| Column Name | Data Type |
|------------|-----------|
| product_id | INT |
| category | VARCHAR |
| product_name | VARCHAR |

### manufacture_sales

| Column Name | Data Type |
|------------|-----------|
| sale_id | INT |
| product_id | INT |
| quantity | INT |
| revenue | DECIMAL |

## Requirements

- Include only products that have matching sales records.
- Revenue should be rounded to the nearest whole number.
- Determine rankings within each category.
- Products with the same revenue share the same rank.
- Ranking positions may be skipped after ties.
- Return results sorted by category and rank.
- Return results matching the required output schema and order.

## Output Columns

| Column Name |
|------------|
| category |
| product_name |
| rank |
| revenue |

## Sample Input

### manufacture_product

| product_id | category | product_name |
|------------|----------|--------------|
| 1 | A | Product1 |
| 2 | A | Product2 |
| 3 | A | Product3 |
| 4 | B | Product4 |
| 5 | B | Product5 |
| 6 | B | Product6 |

### manufacture_sales

| sale_id | product_id | quantity | revenue |
|----------|------------|----------|---------|
| 1 | 1 | 12 | 180 |
| 2 | 2 | 8 | 120 |
| 3 | 3 | 10 | 120 |
| 4 | 4 | 7 | 69.6 |
| 6 | 6 | 5 | 50 |

## Sample Output

| category | product_name | rank | revenue |
|----------|--------------|------|---------|
| A | Product1 | 1 | 180 |
| A | Product2 | 2 | 120 |
| A | Product3 | 2 | 120 |
| B | Product4 | 1 | 70 |
| B | Product6 | 2 | 50 |

## Expected Output Schema

| Column Name | Data Type |
|------------|-----------|
| category | STRING |
| product_name | STRING |
| rank | INT |
| revenue | INT |

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *
from pyspark.sql import Window

# manufacture_product
manufacture_product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("category", StringType(), True),
    StructField("product_name", StringType(), True)
])

manufacture_product_data = [
    (1, "A", "Product1"),
    (2, "A", "Product2"),
    (3, "A", "Product3"),
    (4, "B", "Product4"),
    (5, "B", "Product5"),
    (6, "B", "Product6")
]

manufacture_product_df = spark.createDataFrame(
    manufacture_product_data,
    schema=manufacture_product_schema
)

# manufacture_sales
manufacture_sales_schema = StructType([
    StructField("sale_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("revenue", DoubleType(), True)
])

manufacture_sales_data = [
    (1, 1, 12, 180.0),
    (2, 2, 8, 120.0),
    (3, 3, 10, 120.0),
    (4, 4, 7, 69.6),
    (6, 6, 5, 50.0)
]

manufacture_sales_df = spark.createDataFrame(
    manufacture_sales_data,
    schema=manufacture_sales_schema
)

In [0]:
result_df = (
    manufacture_product_df.join(manufacture_sales_df, on="product_id", how="inner")
    .withColumn(
        "rank", rank().over(Window.partitionBy("category").orderBy(desc("revenue")))
    )
    .select(
        col("category"),
        col("product_name"),
        col("rank"),
        ceil(col("revenue")).alias("revenue"),
    )
)
display(result_df)